# Paper 1 — Controlled 2×2 Ablation
## Conformal-Time / First-Integral vs Affine / Second-Order, with and without RAR

This notebook is the **controlled factorial ablation** for the numerical core of Paper 1.

### The scientific question

The enhanced formulation has changed two things relative to the earlier baseline:

1. **Mathematical representation**
   - Old: affine parameter \(\lambda\) as the computational variable, learning \(\eta(\lambda),x(\lambda)\) from the raw second-order geodesic equations.
   - New: conformal time \(\eta\) as the computational variable, learning \(x(\eta),\lambda(\eta)\) from the reduced first-integral equations.

2. **Sampling strategy**
   - RAR on: residual-adaptive refinement of the collocation set.
   - RAR off: a matched fixed-grid control with the **same number of collocation points at every training stage** as the RAR run has at that stage.

The 2×2 design is therefore:

| Representation | RAR | Meaning |
|---|---|---|
| Affine second-order | OFF | baseline representation + fixed sampling |
| Affine second-order | ON | baseline representation + adaptive sampling |
| Conformal first-integral | OFF | new representation + fixed sampling |
| Conformal first-integral | ON | new representation + adaptive sampling |

The critical design choice is that **all four runs use the same physical problem, same seed, same network architecture, same initial network weights, same Adam budget, same L-BFGS budget, same stage schedule, and matched collocation counts**. Only the two experimental factors change.

### Important interpretation rule

This is a **factor-isolation experiment**, not a five-seed robustness experiment. The earlier five-seed conformal+RAR matter study already tests initialization stability of the proposed formulation. Here we use one fixed seed to isolate the effects of representation and RAR. If the 2×2 effect is ambiguous, an optional second-seed confirmation can be enabled at the top of the notebook.

### Recommended Colab workflow

1. Run the notebook once in `PROFILE = "smoke"` mode.
2. If the smoke test completes normally, restart the Colab runtime.
3. Change exactly one line to `PROFILE = "paper"`.
4. Run the entire notebook once.
5. Download the generated `paper1_results/..._2x2_ablation` directory as a ZIP using the final export cell.

**Do not delete previous `paper1_results/` directories.** Each run receives a timestamped directory.

### Final fairness control added in this version

Because the affine benchmark begins with relatively large physical derivatives, this version adds a separate **affine-specific nondimensionalization control**. It uses the same raw second-order geodesic equations, the same network, initialization, optimizer budget, validation grid, and RAR conditions, but scales the affine coordinate and output variables so that the prescribed initial slopes are order unity. If the conformal formulation still retains a large advantage over this better-conditioned affine control, the numerical evidence for a representation-level effect is substantially stronger.


In [ ]:
# ============================================================
# 0. CONTROL PANEL — CHANGE ONLY THESE SETTINGS
# ============================================================

# SAFE DEFAULT: smoke test.
# After the smoke test succeeds in a fresh runtime, change to "paper".
PROFILE = "paper"          # "smoke" or "paper"
SEED = 42
RUN_SECOND_SEED_CONFIRMATION = False   # optional; normally leave False
RUN_AFFINE_FAIRNESS = True                 # add affine-specific nondimensionalization controls

# Physical matter benchmark — kept identical to the validated Paper 1 setup.
MATTER_ETA0 = 0.5
MATTER_X0 = 0.0
MATTER_P = 1.0
MATTER_KAPPA = 1.0
MATTER_A = 1.0
MATTER_LAMBDA0 = 0.0
MATTER_LAMBDA_MAX = 0.8

# Network: IDENTICAL architecture for all four cells of the 2×2 design.
WIDTH = 64
DEPTH = 5

# Optimization: IDENTICAL budget for all four cells.
ADAM_LR = 1e-3
LBFGS_LR = 0.5
LBFGS_HISTORY = 50

PROFILES = {
    "smoke": {
        "ADAM_EPOCHS_PER_STAGE": 60,
        "N_INITIAL": 96,
        "N_ADD_PER_STAGE": 24,
        "N_STAGES": 2,          # stage 0 + 2 refinements => 144 final points
        "N_CANDIDATE": 600,
        "LBFGS_ITERS": 20,
        "N_VALIDATION": 1200,
    },
    "paper": {
        "ADAM_EPOCHS_PER_STAGE": 1400,
        "N_INITIAL": 256,
        "N_ADD_PER_STAGE": 64,
        "N_STAGES": 4,          # stage 0 + 4 refinements => 512 final points
        "N_CANDIDATE": 5000,
        "LBFGS_ITERS": 220,
        "N_VALIDATION": 4000,
    },
}

if PROFILE not in PROFILES:
    raise ValueError("PROFILE must be 'smoke' or 'paper'.")
CFG = PROFILES[PROFILE]

print("PROFILE:", PROFILE)
print("SEED:", SEED)
print("CONFIG:", CFG)

PROFILE: paper
SEED: 42
CONFIG: {'ADAM_EPOCHS_PER_STAGE': 1400, 'N_INITIAL': 256, 'N_ADD_PER_STAGE': 64, 'N_STAGES': 4, 'N_CANDIDATE': 5000, 'LBFGS_ITERS': 220, 'N_VALIDATION': 4000}


In [ ]:
# ============================================================
# 1. IMPORTS / REPRODUCIBILITY / RESULT MANAGEMENT
# ============================================================

import gc
import json
import math
import shutil
import time
import uuid
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from IPython.display import display
from scipy.integrate import solve_ivp
from scipy.interpolate import PchipInterpolator

DTYPE = torch.float64
DEVICE = torch.device("cpu")
torch.set_default_dtype(DTYPE)
try:
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

# Each notebook execution gets a new run directory.
RESULTS_ROOT = Path("paper1_results")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
RUN_DIR = RESULTS_ROOT / f"{RUN_ID}_2x2_ablation"
RUN_DIR.mkdir(parents=True, exist_ok=False)
FIG_DIR = RUN_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("Run directory:", RUN_DIR.resolve())
print("Device:", DEVICE)
print("dtype:", torch.get_default_dtype())


def seed_everything(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass


# Registry uses one stable schema for every experiment run.
REGISTRY_COLUMNS = [
    "status", "profile", "seed", "representation", "rar", "x_eta_rel_L2",
    "lambda_eta_rel_L2", "eta_lambda_rel_L2", "x_lambda_rel_L2",
    "max_R_eta", "max_R_x", "max_R_lambda", "max_momentum_residual",
    "max_kappa_residual", "min_abs_dlambda_deta", "wall_seconds",
    "collocation_points_final", "adam_epochs_total", "lbfgs_iters",
    "reference_raw_vs_reduced_eta_max", "reference_raw_vs_reduced_x_max",
    "run_id", "created_utc",
]


def append_registry_row(row):
    registry_path = RESULTS_ROOT / "run_registry.csv"
    row = dict(row)
    row["run_id"] = RUN_ID
    row["created_utc"] = RUN_ID.split("_")[0]
    clean = {k: row.get(k, np.nan) for k in REGISTRY_COLUMNS}
    df = pd.DataFrame([clean], columns=REGISTRY_COLUMNS)
    header = not registry_path.exists()
    df.to_csv(registry_path, mode="a", header=header, index=False)


with open(RUN_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "run_id": RUN_ID,
        "created_utc": RUN_ID.split("_")[0],
        "experiment": "2x2_ablation",
        "profile": PROFILE,
        "seed": SEED,
        "second_seed_confirmation": RUN_SECOND_SEED_CONFIRMATION,
        "network": {"width": WIDTH, "depth": DEPTH},
        "optimizer": {
            "adam_lr": ADAM_LR,
            "lbfgs_lr": LBFGS_LR,
            "lbfgs_history": LBFGS_HISTORY,
            "lbfgs_iters": CFG["LBFGS_ITERS"],
        },
        "training": CFG,
        "matter": {
            "A": MATTER_A, "eta0": MATTER_ETA0, "x0": MATTER_X0,
            "p": MATTER_P, "kappa": MATTER_KAPPA,
            "lambda0": MATTER_LAMBDA0, "lambda_max": MATTER_LAMBDA_MAX,
        },
    }, f, indent=2)

Run ID: 20260925T230536Z_c8ce5c
Run directory: /content/paper1_results/20260925T230536Z_c8ce5c_2x2_ablation
Device: cpu
dtype: torch.float64


## 2. The physical problem and the independent reference

We use the same validated matter benchmark as the earlier Paper 1 experiments:

\[
 a(\eta)=A_m\eta^2,
\qquad
\eta_0=0.5,
\qquad
p=1,
\qquad
\kappa=1,
\qquad
\lambda_0=0,
\qquad
\lambda_{\max}=0.8.
\]

The independent reference is **not** the reduced PINN equations. It is obtained by integrating the raw affine-parameter geodesic system directly:

\[
\eta''+\frac{a'}a(\eta'^2+x'^2)=0,
\]

\[
x''+2\frac{a'}a\eta'x'=0.
\]

A separate DOP853 solve of the first-integral system is then used only as a consistency check against the raw reference. The PINN's errors are reported against the raw second-order DOP853 trajectory.

In [ ]:
# ============================================================
# 2. PROBLEM DEFINITIONS + INDEPENDENT REFERENCES
# ============================================================

@dataclass(frozen=True)
class Problem:
    name: str
    A: float
    eta0: float
    x0: float
    p: float
    kappa: float
    lambda0: float
    lambda_max: float

MATTER = Problem(
    "matter", MATTER_A, MATTER_ETA0, MATTER_X0,
    MATTER_P, MATTER_KAPPA, MATTER_LAMBDA0, MATTER_LAMBDA_MAX
)


def scale_factor_np(eta, problem=MATTER):
    eta = np.asarray(eta, dtype=float)
    if problem.name == "matter":
        return problem.A * eta**2
    raise ValueError(problem.name)


def scale_factor_torch(eta, problem=MATTER):
    if problem.name == "matter":
        return problem.A * eta**2
    raise ValueError(problem.name)


def relative_l2(pred, truth):
    pred = np.asarray(pred)
    truth = np.asarray(truth)
    return float(np.linalg.norm(pred - truth) / (np.linalg.norm(truth) + 1e-15))


def norm_max(x):
    return float(np.max(np.abs(np.asarray(x))))


def initial_affine_state(problem):
    a0 = problem.A * problem.eta0**2
    x0p = problem.p / a0**2
    eta0p = np.sqrt(problem.p**2 + problem.kappa * a0**2) / a0**2
    return eta0p, x0p


def raw_geodesic_system(lam, y, problem):
    eta, x, eta_p, x_p = y
    a = problem.A * eta**2
    a_prime = 2.0 * problem.A * eta
    Hc = a_prime / a
    eta_pp = -Hc * (eta_p**2 + x_p**2)
    x_pp = -2.0 * Hc * eta_p * x_p
    return [eta_p, x_p, eta_pp, x_pp]


def solve_raw_second_order(problem, n_dense=None, rtol=1e-11, atol=1e-11):
    eta0p, x0p = initial_affine_state(problem)
    y0 = [problem.eta0, problem.x0, eta0p, x0p]
    sol = solve_ivp(
        lambda lam, y: raw_geodesic_system(lam, y, problem),
        (problem.lambda0, problem.lambda_max),
        y0,
        method="DOP853",
        rtol=rtol,
        atol=atol,
        dense_output=True,
    )
    if not sol.success:
        raise RuntimeError(sol.message)
    return sol


def first_integral_eta_system(eta, y, problem):
    x, lam = y
    a = problem.A * eta**2
    den = np.sqrt(problem.p**2 + problem.kappa * a**2)
    dx = problem.p / den
    dl = a**2 / den
    return [dx, dl]


def build_reference_bundle(problem=MATTER, n=4000):
    raw = solve_raw_second_order(problem)
    lam_dense = np.linspace(problem.lambda0, problem.lambda_max, max(n, 5000))
    raw_vals = raw.sol(lam_dense)
    eta_raw, x_raw = raw_vals[0], raw_vals[1]

    # The raw matter trajectory is monotone in eta for this branch.
    eta_end = float(eta_raw[-1])
    eta_grid = np.linspace(problem.eta0, eta_end, n)

    # Independent reduced reference — used only as a consistency check.
    red = solve_ivp(
        lambda eta, y: first_integral_eta_system(eta, y, problem),
        (problem.eta0, eta_end),
        [problem.x0, problem.lambda0],
        method="DOP853",
        rtol=1e-12,
        atol=1e-12,
        dense_output=True,
    )
    x_reduced, lam_reduced = red.sol(eta_grid)

    # Raw reference represented as functions of eta.
    order = np.argsort(eta_raw)
    eta_sorted = eta_raw[order]
    x_sorted = x_raw[order]
    lam_sorted = lam_dense[order]
    eta_unique, idx = np.unique(eta_sorted, return_index=True)
    x_unique = x_sorted[idx]
    lam_unique = lam_sorted[idx]
    x_of_eta = PchipInterpolator(eta_unique, x_unique)
    lam_of_eta = PchipInterpolator(eta_unique, lam_unique)

    # Raw-vs-reduced consistency on the common eta grid.
    x_raw_on_eta = x_of_eta(eta_grid)
    lam_raw_on_eta = lam_of_eta(eta_grid)
    raw_reduced_eta_max = norm_max(eta_grid - eta_grid)  # exactly zero by construction
    raw_reduced_x_max = norm_max(x_raw_on_eta - x_reduced)
    raw_reduced_lambda_max = norm_max(lam_raw_on_eta - lam_reduced)

    return {
        "raw": raw,
        "eta_grid": eta_grid,
        "x_of_eta": x_of_eta,
        "lambda_of_eta": lam_of_eta,
        "x_reduced": x_reduced,
        "lambda_reduced": lam_reduced,
        "raw_reduced_eta_max_abs": raw_reduced_eta_max,
        "raw_reduced_x_max_abs": raw_reduced_x_max,
        "raw_reduced_lambda_max_abs": raw_reduced_lambda_max,
    }


print("Building independent raw + reduced reference ...")
REF = build_reference_bundle(MATTER, n=CFG["N_VALIDATION"])
ETA_MIN = float(REF["eta_grid"][0])
ETA_MAX = float(REF["eta_grid"][-1])
print("Matter eta range:", ETA_MIN, "to", ETA_MAX)
print("Raw vs reduced x max abs:", REF["raw_reduced_x_max_abs"])
print("Raw vs reduced lambda max abs:", REF["raw_reduced_lambda_max_abs"])

Building independent raw + reduced reference ...
Matter eta range: 0.5 to 1.4876770670813468
Raw vs reduced x max abs: 9.582000990186398e-10
Raw vs reduced lambda max abs: 1.9836060754428687e-09


## 3A. Final fairness check: affine-specific nondimensionalization

The original affine baseline already uses a scaled network input and hard initial conditions, but its **physical affine derivatives are large at the initial point** for the chosen benchmark:

\[
\eta'_0 = \frac{\sqrt{p^2+\kappa a_0^2}}{a_0^2},\qquad
x'_0=\frac{p}{a_0^2}.
\]

For the matter benchmark used here, these are of order \(10\). A skeptical reader could therefore ask whether the large affine error is partly a conditioning artifact that could be reduced simply by nondimensionalizing the affine problem.

This notebook therefore adds a **fairness-control pair** without changing the underlying second-order geodesic equations. The physical predictions remain \(\eta(\lambda)\) and \(x(\lambda)\), but the neural network learns dimensionless variables

\[
\widehat\eta = \frac{\eta-\eta_0}{S_\eta},\qquad
\widehat x = \frac{x-x_0}{S_x},
\]

on a normalized affine coordinate

\[
\tau=\frac{\lambda-\lambda_0}{\Delta\lambda}.
\]

The scales are fixed **before training** from the prescribed initial state,

\[
S_\eta=|\eta'_0|\,\Delta\lambda,\qquad
S_x=|x'_0|\,\Delta\lambda,
\]

and the second-order residuals are expressed in the corresponding dimensionless form. This is a mathematical rescaling of the same affine equations, not a change of physical model or target trajectory.

The fairness control is run both **without RAR and with RAR**, so that the effect of nondimensionalization can be separated from the effect of sampling.


## 3. Shared neural architecture and identical initialization

Both representations use the same fully-connected \(1\rightarrow64\rightarrow64\rightarrow64\rightarrow64\rightarrow64\rightarrow2\) Tanh network.

The input is linearly scaled to \([-1,1]\). Initial conditions are imposed analytically through a quadratic correction, so the network starts by satisfying the initial position and first derivative constraints exactly.

A single initial state dictionary is generated from the fixed seed and then copied into all four factor cells. This prevents the ablation from accidentally becoming an initialization experiment.

In [ ]:
# ============================================================
# 3. SHARED NETWORK UTILITIES
# ============================================================

def build_mlp(in_dim=1, out_dim=2, width=64, depth=5):
    layers = [nn.Linear(in_dim, width), nn.Tanh()]
    for _ in range(depth - 1):
        layers += [nn.Linear(width, width), nn.Tanh()]
    layers += [nn.Linear(width, out_dim)]
    net = nn.Sequential(*layers)
    for layer in net:
        if isinstance(layer, nn.Linear):
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)
    return net


def make_shared_initial_state(seed):
    seed_everything(seed)
    probe = build_mlp(1, 2, WIDTH, DEPTH)
    state = {k: v.detach().clone() for k, v in probe.state_dict().items()}
    del probe
    return state


SHARED_INIT_STATE = make_shared_initial_state(SEED)
print("Shared initialization captured from seed", SEED)

Shared initialization captured from seed 42


## 4. Representation A — affine-parameter, raw second-order PINN

The baseline predicts

\[
\lambda\mapsto(\eta_\theta(\lambda),x_\theta(\lambda))
\]

and minimizes

\[
R_\eta=\eta''+\frac{a'}a(\eta'^2+x'^2),
\]

\[
R_x=x''+2\frac{a'}a\eta'x'.
\]

For matter, \(a=A\eta^2\), so \(a'/a=2/\eta\).

This is the exact baseline representation we want to compare with the conformal first-integral method.

In [ ]:
# ============================================================
# 4. AFFINE SECOND-ORDER MODEL
# ============================================================

def deriv(y, x, create_graph=True):
    return torch.autograd.grad(
        y, x,
        grad_outputs=torch.ones_like(y),
        create_graph=create_graph,
        retain_graph=True,
    )[0]


class AffineSecondOrderPINN(nn.Module):
    def __init__(self, problem, lam_min, lam_max, width=64, depth=5, init_state=None):
        super().__init__()
        self.problem = problem
        self.lam_min = float(lam_min)
        self.lam_max = float(lam_max)
        self.net = build_mlp(1, 2, width, depth)
        if init_state is not None:
            self.net.load_state_dict(init_state)

        self.eta0p, self.x0p = initial_affine_state(problem)

    def forward(self, lam):
        lam = lam.to(device=DEVICE, dtype=DTYPE)
        s = 2.0 * (lam - self.lam_min) / (self.lam_max - self.lam_min) - 1.0
        raw = self.net(s)
        d = lam - self.problem.lambda0
        eta = self.problem.eta0 + self.eta0p * d + d**2 * raw[:, 0:1]
        x = self.problem.x0 + self.x0p * d + d**2 * raw[:, 1:2]
        return eta, x


def affine_second_order_residuals(model, lam):
    lam = lam.detach().clone().requires_grad_(True)
    eta, x = model(lam)
    eta_p = deriv(eta, lam)
    x_p = deriv(x, lam)
    eta_pp = deriv(eta_p, lam)
    x_pp = deriv(x_p, lam)

    a = scale_factor_torch(eta, model.problem)
    Hc = 2.0 / eta  # matter: a'/a = 2/eta

    R_eta = eta_pp + Hc * (eta_p**2 + x_p**2)
    R_x = x_pp + 2.0 * Hc * eta_p * x_p
    return {
        "lambda": lam,
        "eta": eta,
        "x": x,
        "eta_p": eta_p,
        "x_p": x_p,
        "R_eta": R_eta,
        "R_x": R_x,
    }


def affine_loss(model, lam_grid):
    r = affine_second_order_residuals(model, lam_grid)
    L_eta = torch.mean(r["R_eta"]**2)
    L_x = torch.mean(r["R_x"]**2)
    return r, L_eta + L_x, L_eta, L_x

In [ ]:
# ============================================================
# 4A. AFFINE SECOND-ORDER MODEL WITH EXPLICIT NONDIMENSIONALIZATION
# ============================================================

class AffinePreconditionedSecondOrderPINN(nn.Module):
    """
    Same physical second-order affine geodesic equations as the baseline,
    but written in dimensionless affine/output variables.

    This is a fairness control, not a new physical model.
    """
    def __init__(self, problem, lam_min, lam_max, width=64, depth=5, init_state=None):
        super().__init__()
        self.problem = problem
        self.lam_min = float(lam_min)
        self.lam_max = float(lam_max)
        self.delta_lam = float(lam_max - lam_min)
        self.net = build_mlp(1, 2, width, depth)
        if init_state is not None:
            self.net.load_state_dict(init_state)

        self.eta0p, self.x0p = initial_affine_state(problem)

        # Scales are determined only by known initial data and the affine interval.
        # They make the leading normalized slopes O(1) without using the reference solution.
        self.eta_scale = max(abs(self.eta0p) * self.delta_lam, 1e-12)
        self.x_scale = max(abs(self.x0p) * self.delta_lam, 1e-12)
        self.eta_slope_tau = self.eta0p * self.delta_lam / self.eta_scale
        self.x_slope_tau = self.x0p * self.delta_lam / self.x_scale

    def forward(self, lam):
        lam = lam.to(device=DEVICE, dtype=DTYPE)
        tau = (lam - self.lam_min) / self.delta_lam
        s = 2.0 * tau - 1.0
        raw = self.net(s)

        # Hard initial position and first derivative constraints in normalized variables.
        eta_hat = self.eta_slope_tau * tau + tau**2 * raw[:, 0:1]
        x_hat = self.x_slope_tau * tau + tau**2 * raw[:, 1:2]
        eta = self.problem.eta0 + self.eta_scale * eta_hat
        x = self.problem.x0 + self.x_scale * x_hat
        return eta, x


def affine_preconditioned_residuals(model, lam):
    lam = lam.detach().clone().requires_grad_(True)
    eta, x = model(lam)
    eta_p = deriv(eta, lam)
    x_p = deriv(x, lam)
    eta_pp = deriv(eta_p, lam)
    x_pp = deriv(x_p, lam)

    Hc = 2.0 / eta  # matter: a'/a = 2/eta
    R_eta_phys = eta_pp + Hc * (eta_p**2 + x_p**2)
    R_x_phys = x_pp + 2.0 * Hc * eta_p * x_p

    # Dimensionless residuals. The zero set is unchanged; only the optimization scale changes.
    R_eta = (model.delta_lam**2 / model.eta_scale) * R_eta_phys
    R_x = (model.delta_lam**2 / model.x_scale) * R_x_phys

    return {
        "lambda": lam,
        "eta": eta,
        "x": x,
        "eta_p": eta_p,
        "x_p": x_p,
        "R_eta": R_eta,
        "R_x": R_x,
        "R_eta_phys": R_eta_phys,
        "R_x_phys": R_x_phys,
    }


def affine_preconditioned_loss(model, lam_grid):
    r = affine_preconditioned_residuals(model, lam_grid)
    L_eta = torch.mean(r["R_eta"]**2)
    L_x = torch.mean(r["R_x"]**2)
    return r, L_eta + L_x, L_eta, L_x


## 5. Representation B — conformal-time, first-integral PINN

The enhanced model predicts

\[
\eta\mapsto(x_\theta(\eta),\lambda_\theta(\eta)).
\]

The loss is built from

\[
R_x=\frac{dx_\theta}{d\eta}
-\frac{p}{\sqrt{p^2+\kappa a(\eta)^2}},
\]

and

\[
R_\lambda=\frac{d\lambda_\theta}{d\eta}
-\frac{a(\eta)^2}{\sqrt{p^2+\kappa a(\eta)^2}}.
\]

This is the geometry-aware representation whose numerical behavior is being tested by the ablation.

In [ ]:
# ============================================================
# 5. CONFORMAL FIRST-INTEGRAL MODEL
# ============================================================

class ConformalFirstIntegralPINN(nn.Module):
    def __init__(self, problem, eta_min, eta_max, width=64, depth=5, init_state=None):
        super().__init__()
        self.problem = problem
        self.eta_min = float(eta_min)
        self.eta_max = float(eta_max)
        self.net = build_mlp(1, 2, width, depth)
        if init_state is not None:
            self.net.load_state_dict(init_state)

        a0 = problem.A * problem.eta0**2
        den0 = np.sqrt(problem.p**2 + problem.kappa * a0**2)
        self.dx0 = float(problem.p / den0)
        self.dlambda0 = float(a0**2 / den0)

    def forward(self, eta):
        eta = eta.to(device=DEVICE, dtype=DTYPE)
        s = 2.0 * (eta - self.eta_min) / (self.eta_max - self.eta_min) - 1.0
        raw = self.net(s)
        d = eta - self.problem.eta0
        x = self.problem.x0 + self.dx0 * d + d**2 * raw[:, 0:1]
        lam = self.problem.lambda0 + self.dlambda0 * d + d**2 * raw[:, 1:2]
        return x, lam


def conformal_residuals(model, eta):
    eta = eta.detach().clone().requires_grad_(True)
    x, lam = model(eta)
    dx = deriv(x, eta)
    dl = deriv(lam, eta)
    a = scale_factor_torch(eta, model.problem)
    den = torch.sqrt(model.problem.p**2 + model.problem.kappa * a**2)
    target_x = model.problem.p / den
    target_l = a**2 / den
    return {
        "eta": eta,
        "x": x,
        "lambda": lam,
        "dx": dx,
        "dlambda": dl,
        "R_x": dx - target_x,
        "R_lambda": dl - target_l,
    }


def conformal_loss(model, eta_grid):
    r = conformal_residuals(model, eta_grid)
    Lx = torch.mean(r["R_x"]**2)
    Ll = torch.mean(r["R_lambda"]**2)
    return r, Lx + Ll, Lx, Ll

## 6. Matched fixed-vs-adaptive sampling

To isolate RAR, the OFF and ON conditions receive the same number of collocation points at every stage.

For the paper profile:

\[
256\rightarrow320\rightarrow384\rightarrow448\rightarrow512.
\]

- **RAR ON:** after each stage, the next 64 points are selected from the largest residuals on a candidate grid.
- **RAR OFF:** after each stage, the next grid is simply a deterministic Chebyshev grid with the same point count.

Both conditions therefore see the same number of points and the same total Adam budget. The only difference is **where the points are placed**.

The final L-BFGS polishing stage is also identical.

In [ ]:
# ============================================================
# 6. COLLOCATION GRIDS + RAR ENGINE
# ============================================================

def chebyshev_grid(vmin, vmax, n):
    j = np.arange(n)
    nodes = np.cos(np.pi * (2*j + 1) / (2*n))
    vals = vmin + 0.5 * (1.0 - nodes) * (vmax - vmin)
    vals = np.concatenate([[vmin, vmax], vals])
    vals = np.unique(np.sort(vals))
    return torch.tensor(vals.reshape(-1,1), dtype=DTYPE, device=DEVICE)


def candidate_scores(model, candidates, residual_fn, chunk=800):
    scores = []
    for start in range(0, len(candidates), chunk):
        part = candidates[start:start+chunk]
        r = residual_fn(model, part)
        vals = []
        if "R_eta" in r and "R_x" in r:
            residual_keys = ("R_eta", "R_x")
        else:
            residual_keys = ("R_x", "R_lambda")
        for key in residual_keys:
            vals.append(r[key].detach().cpu().numpy()**2)
        scores.append(np.sum(vals, axis=0).reshape(-1))
    return np.concatenate(scores)


def refine_grid(model, current, vmin, vmax, n_candidate, n_add, residual_fn):
    candidates = torch.linspace(vmin, vmax, n_candidate, dtype=DTYPE, device=DEVICE).reshape(-1,1)
    scores = candidate_scores(model, candidates, residual_fn)

    # Exclude candidate points that are already in the current collocation set.
    current_np = current.detach().cpu().numpy().reshape(-1)
    cand_np = candidates.detach().cpu().numpy().reshape(-1)
    available = np.ones(len(cand_np), dtype=bool)
    for value in current_np:
        available &= np.abs(cand_np - value) > 1e-12

    ranked = np.argsort(scores[available])[::-1]
    available_idx = np.where(available)[0]
    take = min(n_add, len(ranked))
    idx = available_idx[ranked[:take]]
    new_points = cand_np[idx]

    merged = np.concatenate([current_np, new_points])
    merged = np.unique(np.sort(merged))
    new_grid = torch.tensor(merged.reshape(-1,1), dtype=DTYPE, device=DEVICE)
    return new_grid, float(scores.max())


def matched_training_stages(vmin, vmax):
    # Exactly the point counts used by the RAR-enabled run.
    counts = [CFG["N_INITIAL"] + i * CFG["N_ADD_PER_STAGE"] for i in range(CFG["N_STAGES"] + 1)]
    return [chebyshev_grid(vmin, vmax, int(n)) for n in counts]

In [ ]:
# ============================================================
# 7. OPTIMIZATION LOOPS — SAME BUDGET FOR ALL FOUR CELLS
# ============================================================

def run_adam_stages(model, initial_grid, vmin, vmax, residual_fn, loss_fn,
                    rar_on, verbose=True):
    current = initial_grid.detach().clone()
    history = []
    stage_records = []
    fixed_grids = matched_training_stages(vmin, vmax) if not rar_on else None

    total_stages = CFG["N_STAGES"] + 1
    total_adam_epochs = CFG["ADAM_EPOCHS_PER_STAGE"] * total_stages

    for stage in range(total_stages):
        if not rar_on and stage > 0:
            # Matched fixed-grid control: same point count as RAR condition,
            # but the points are chosen deterministically rather than by residual.
            current = fixed_grids[stage]

        opt = optim.Adam(model.parameters(), lr=ADAM_LR)
        for epoch in range(CFG["ADAM_EPOCHS_PER_STAGE"]):
            opt.zero_grad(set_to_none=True)
            r, L, L1, L2 = loss_fn(model, current)
            if not torch.isfinite(L):
                raise FloatingPointError(
                    f"Non-finite loss in stage={stage}, epoch={epoch+1}. "
                    "Inspect the saved partial run and do not interpret this condition as success."
                )
            L.backward()
            opt.step()
            history.append({
                "stage": stage,
                "epoch_in_stage": epoch + 1,
                "epoch_global": stage * CFG["ADAM_EPOCHS_PER_STAGE"] + epoch + 1,
                "loss": float(L.item()),
                "L1": float(L1.item()),
                "L2": float(L2.item()),
                "collocation_points": int(len(current)),
                "rar_on": bool(rar_on),
            })

            if verbose and (epoch + 1) in {
                1,
                CFG["ADAM_EPOCHS_PER_STAGE"] // 2,
                CFG["ADAM_EPOCHS_PER_STAGE"],
            }:
                print(
                    f"stage={stage} epoch={epoch+1}/{CFG['ADAM_EPOCHS_PER_STAGE']} "
                    f"points={len(current)} loss={L.item():.3e}"
                )

        if rar_on and stage < CFG["N_STAGES"]:
            current, max_res = refine_grid(
                model,
                current,
                vmin, vmax,
                CFG["N_CANDIDATE"],
                CFG["N_ADD_PER_STAGE"],
                residual_fn,
            )
            stage_records.append({
                "stage": stage + 1,
                "action": "RAR_add",
                "collocation_points": int(len(current)),
                "candidate_max_residual": max_res,
            })
        else:
            stage_records.append({
                "stage": stage,
                "action": "fixed_grid" if not rar_on else "final",
                "collocation_points": int(len(current)),
            })

    # Exactly the same LBFGS configuration for all four cells.
    lbfgs = optim.LBFGS(
        model.parameters(),
        lr=LBFGS_LR,
        max_iter=CFG["LBFGS_ITERS"],
        history_size=LBFGS_HISTORY,
        line_search_fn="strong_wolfe",
    )

    def closure():
        lbfgs.zero_grad(set_to_none=True)
        _, L, _, _ = loss_fn(model, current)
        if not torch.isfinite(L):
            raise FloatingPointError("Non-finite L-BFGS loss.")
        L.backward()
        return L

    lbfgs.step(closure)
    return model, current, pd.DataFrame(history), pd.DataFrame(stage_records), total_adam_epochs

## 7. Validation

All four models are evaluated on dense **held-out** grids after training.

The principal comparison is made in the common affine representation

\[
\eta(\lambda),\qquad x(\lambda),
\]

because this is the trajectory representation shared by the original and new formulations.

For the conformal model, \(\lambda(\eta)\) is inverted with monotone PCHIP interpolation and the trajectory is reconstructed onto the same affine-parameter grid used by the raw DOP853 reference.

We also report the native errors:

- affine model: \(\eta(\lambda),x(\lambda)\);
- conformal model: \(x(\eta),\lambda(\eta)\).

Invariant diagnostics are reported for both formulations:

\[
p_{\rm learned}=a^2\frac{dx}{d\lambda},
\qquad
\kappa_{\rm learned}=a^2\left[\left(\frac{d\eta}{d\lambda}\right)^2-\left(\frac{dx}{d\lambda}\right)^2\right].
\]

In [ ]:
# ============================================================
# 8. VALIDATION HELPERS
# ============================================================

def validate_affine(model, problem, ref, n_eval, residual_fn=affine_second_order_residuals):
    lam_eval = np.linspace(problem.lambda0, problem.lambda_max, n_eval)
    lam_t = torch.tensor(lam_eval.reshape(-1,1), dtype=DTYPE, device=DEVICE)
    with torch.enable_grad():
        r = residual_fn(model, lam_t)

    eta_pred = r["eta"].detach().cpu().numpy().reshape(-1)
    x_pred = r["x"].detach().cpu().numpy().reshape(-1)
    eta_p = r["eta_p"].detach().cpu().numpy().reshape(-1)
    x_p = r["x_p"].detach().cpu().numpy().reshape(-1)
    R_eta = r["R_eta"].detach().cpu().numpy().reshape(-1)
    R_x = r["R_x"].detach().cpu().numpy().reshape(-1)

    eta_ref, x_ref = ref["raw"].sol(lam_eval)[:2]
    a = scale_factor_np(eta_pred, problem)
    p_res = a**2 * x_p - problem.p
    k_res = a**2 * (eta_p**2 - x_p**2) - problem.kappa

    metrics = {
        "x_eta_rel_L2": np.nan,
        "lambda_eta_rel_L2": np.nan,
        "eta_lambda_rel_L2": relative_l2(eta_pred, eta_ref),
        "x_lambda_rel_L2": relative_l2(x_pred, x_ref),
        "max_R_eta": norm_max(R_eta),
        "max_R_x": norm_max(R_x),
        "max_R_lambda": np.nan,
        "max_momentum_residual": norm_max(p_res),
        "max_kappa_residual": norm_max(k_res),
        "min_abs_dlambda_deta": np.nan,
    }
    arrays = {
        "lambda": lam_eval,
        "eta_pred": eta_pred,
        "x_pred": x_pred,
        "eta_ref": eta_ref,
        "x_ref": x_ref,
        "R_eta": R_eta,
        "R_x": R_x,
        "eta_prime": eta_p,
        "x_prime": x_p,
        "p_residual": p_res,
        "kappa_residual": k_res,
    }
    return metrics, arrays


def validate_conformal(model, problem, ref, n_eval):
    eta_eval = np.linspace(ETA_MIN, ETA_MAX, n_eval)
    eta_t = torch.tensor(eta_eval.reshape(-1,1), dtype=DTYPE, device=DEVICE)
    with torch.enable_grad():
        r = conformal_residuals(model, eta_t)

    x_pred = r["x"].detach().cpu().numpy().reshape(-1)
    lam_pred = r["lambda"].detach().cpu().numpy().reshape(-1)
    dx_deta = r["dx"].detach().cpu().numpy().reshape(-1)
    dl_deta = r["dlambda"].detach().cpu().numpy().reshape(-1)
    R_x = r["R_x"].detach().cpu().numpy().reshape(-1)
    R_lambda = r["R_lambda"].detach().cpu().numpy().reshape(-1)

    x_ref_eta = ref["x_of_eta"](eta_eval)
    lam_ref_eta = ref["lambda_of_eta"](eta_eval)

    # Need monotonic lambda for a valid inverse reconstruction.
    order = np.argsort(lam_pred)
    lam_sorted = lam_pred[order]
    eta_sorted = eta_eval[order]
    x_sorted = x_pred[order]
    lam_unique, idx = np.unique(lam_sorted, return_index=True)
    eta_unique = eta_sorted[idx]
    x_unique = x_sorted[idx]
    if len(lam_unique) < 10 or np.any(np.diff(lam_unique) <= 0):
        raise RuntimeError("Conformal model produced non-monotone lambda; affine reconstruction is invalid.")

    lam_eval = np.linspace(problem.lambda0, problem.lambda_max, n_eval)
    eta_recon = PchipInterpolator(lam_unique, eta_unique)(lam_eval)
    x_recon = PchipInterpolator(lam_unique, x_unique)(lam_eval)
    eta_ref, x_ref = ref["raw"].sol(lam_eval)[:2]

    dx_dlambda = dx_deta / dl_deta
    eta_dlambda = 1.0 / dl_deta
    a = scale_factor_np(eta_eval, problem)
    p_res = a**2 * dx_dlambda - problem.p
    k_res = a**2 * (eta_dlambda**2 - dx_dlambda**2) - problem.kappa

    metrics = {
        "x_eta_rel_L2": relative_l2(x_pred, x_ref_eta),
        "lambda_eta_rel_L2": relative_l2(lam_pred, lam_ref_eta),
        "eta_lambda_rel_L2": relative_l2(eta_recon, eta_ref),
        "x_lambda_rel_L2": relative_l2(x_recon, x_ref),
        "max_R_eta": np.nan,
        "max_R_x": norm_max(R_x),
        "max_R_lambda": norm_max(R_lambda),
        "max_momentum_residual": norm_max(p_res),
        "max_kappa_residual": norm_max(k_res),
        "min_abs_dlambda_deta": float(np.min(np.abs(dl_deta))),
    }
    arrays = {
        "eta": eta_eval,
        "x_pred": x_pred,
        "lambda_pred": lam_pred,
        "x_ref_eta": x_ref_eta,
        "lambda_ref_eta": lam_ref_eta,
        "eta_recon": eta_recon,
        "x_recon": x_recon,
        "eta_ref_lambda": eta_ref,
        "x_ref_lambda": x_ref,
        "R_x": R_x,
        "R_lambda": R_lambda,
        "dx_deta": dx_deta,
        "dlambda_deta": dl_deta,
        "p_residual": p_res,
        "kappa_residual": k_res,
        "lambda_eval": lam_eval,
    }
    return metrics, arrays

## 8. Run the four cells of the factorial design

The order below is deliberate:

1. affine + fixed sampling;
2. affine + RAR;
3. conformal + fixed sampling;
4. conformal + RAR.

The same initial network state is copied into each run. Each run receives the same number of points at the same stage boundaries. The only experimental factors are therefore **representation** and **sampling adaptivity**.

In [ ]:
# ============================================================
# 9. GENERIC TRAIN / EVALUATE FUNCTION FOR ONE FACTOR CELL
# ============================================================

def train_one_cell(representation, rar_on, seed=SEED, verbose=True):
    seed_everything(seed)

    if representation == "affine_second_order":
        model = AffineSecondOrderPINN(
            MATTER, MATTER.lambda0, MATTER.lambda_max,
            width=WIDTH, depth=DEPTH,
            init_state=SHARED_INIT_STATE,
        ).to(DEVICE)
        vmin, vmax = MATTER.lambda0, MATTER.lambda_max
        grid = chebyshev_grid(vmin, vmax, CFG["N_INITIAL"])
        residual_fn = affine_second_order_residuals
        loss_fn = affine_loss
        validation_fn = lambda m: validate_affine(m, MATTER, REF, CFG["N_VALIDATION"], affine_second_order_residuals)
        fairness_note = "raw_affine"
    elif representation == "affine_preconditioned":
        model = AffinePreconditionedSecondOrderPINN(
            MATTER, MATTER.lambda0, MATTER.lambda_max,
            width=WIDTH, depth=DEPTH,
            init_state=SHARED_INIT_STATE,
        ).to(DEVICE)
        vmin, vmax = MATTER.lambda0, MATTER.lambda_max
        grid = chebyshev_grid(vmin, vmax, CFG["N_INITIAL"])
        residual_fn = affine_preconditioned_residuals
        loss_fn = affine_preconditioned_loss
        validation_fn = lambda m: validate_affine(m, MATTER, REF, CFG["N_VALIDATION"], affine_preconditioned_residuals)
        fairness_note = "affine_nondimensionalized"
    elif representation == "conformal_first_integral":
        model = ConformalFirstIntegralPINN(
            MATTER, ETA_MIN, ETA_MAX,
            width=WIDTH, depth=DEPTH,
            init_state=SHARED_INIT_STATE,
        ).to(DEVICE)
        vmin, vmax = ETA_MIN, ETA_MAX
        grid = chebyshev_grid(vmin, vmax, CFG["N_INITIAL"])
        residual_fn = conformal_residuals
        loss_fn = conformal_loss
        validation_fn = lambda m: validate_conformal(m, MATTER, REF, CFG["N_VALIDATION"])
        fairness_note = "conformal_first_integral"
    else:
        raise ValueError(representation)

    t0 = time.time()
    model, final_grid, history, stage_records, total_adam_epochs = run_adam_stages(
        model, grid, vmin, vmax, residual_fn, loss_fn, rar_on, verbose=verbose
    )
    wall_s = time.time() - t0

    metrics, arrays = validation_fn(model)

    metrics.update({
        "representation": representation,
        "rar": bool(rar_on),
        "fairness_note": fairness_note,
        "seed": seed,
        "wall_seconds": wall_s,
        "collocation_points_final": int(len(final_grid)),
        "adam_epochs_total": int(total_adam_epochs),
        "lbfgs_iters": int(CFG["LBFGS_ITERS"]),
        "reference_raw_vs_reduced_eta_max": REF["raw_reduced_eta_max_abs"],
        "reference_raw_vs_reduced_x_max": REF["raw_reduced_x_max_abs"],
        "reference_raw_vs_reduced_lambda_max": REF["raw_reduced_lambda_max_abs"],
    })

    if representation == "affine_preconditioned":
        metrics.update({
            "eta_scale": float(model.eta_scale),
            "x_scale": float(model.x_scale),
            "delta_lambda": float(model.delta_lam),
            "normalized_eta_initial_slope": float(model.eta_slope_tau),
            "normalized_x_initial_slope": float(model.x_slope_tau),
        })
    else:
        metrics.update({
            "eta_scale": np.nan,
            "x_scale": np.nan,
            "delta_lambda": np.nan,
            "normalized_eta_initial_slope": np.nan,
            "normalized_x_initial_slope": np.nan,
        })

    run_name = f"{representation}__rar_{'on' if rar_on else 'off'}"
    run_dir = RUN_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=False)

    pd.DataFrame([metrics]).to_csv(run_dir / "metrics.csv", index=False)
    history.to_csv(run_dir / "training_history.csv", index=False)
    stage_records.to_csv(run_dir / "stage_records.csv", index=False)
    np.savez(run_dir / "validation_arrays.npz", **arrays)
    torch.save(model.state_dict(), run_dir / "model.pt")

    with open(run_dir / "cell_config.json", "w", encoding="utf-8") as f:
        json.dump({
            "representation": representation,
            "rar_on": bool(rar_on),
            "seed": seed,
            "profile": PROFILE,
            "network": {"width": WIDTH, "depth": DEPTH},
            "training": CFG,
            "shared_initial_state": True,
            "fairness_note": fairness_note,
            "preconditioning_scales": {
                "eta_scale": float(metrics["eta_scale"]) if np.isfinite(metrics["eta_scale"]) else None,
                "x_scale": float(metrics["x_scale"]) if np.isfinite(metrics["x_scale"]) else None,
                "delta_lambda": float(metrics["delta_lambda"]) if np.isfinite(metrics["delta_lambda"]) else None,
            },
        }, f, indent=2)

    # Common affine trajectory plot.
    lam_ref_grid = np.linspace(MATTER.lambda0, MATTER.lambda_max, CFG["N_VALIDATION"])
    eta_ref_grid, x_ref_grid = REF["raw"].sol(lam_ref_grid)[:2]
    if representation in {"affine_second_order", "affine_preconditioned"}:
        lam_plot = arrays["lambda"]
        eta_plot = arrays["eta_pred"]
        x_plot = arrays["x_pred"]
    else:
        lam_plot = arrays["lambda_eval"]
        eta_plot = arrays["eta_recon"]
        x_plot = arrays["x_recon"]

    fig = plt.figure(figsize=(8,5))
    plt.plot(lam_ref_grid, eta_ref_grid, "k--", lw=2, label="raw DOP853")
    plt.plot(lam_plot, eta_plot, label=run_name)
    plt.xlabel(r"$\lambda$"); plt.ylabel(r"$\eta(\lambda)$")
    plt.title(run_name + ": affine trajectory")
    plt.grid(alpha=0.3); plt.legend(); plt.tight_layout()
    fig.savefig(run_dir / "eta_lambda.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    fig = plt.figure(figsize=(8,5))
    plt.plot(lam_ref_grid, x_ref_grid, "k--", lw=2, label="raw DOP853")
    plt.plot(lam_plot, x_plot, label=run_name)
    plt.xlabel(r"$\lambda$"); plt.ylabel(r"$x(\lambda)$")
    plt.title(run_name + ": affine spatial trajectory")
    plt.grid(alpha=0.3); plt.legend(); plt.tight_layout()
    fig.savefig(run_dir / "x_lambda.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    return metrics, arrays, history, stage_records, run_dir

# The original 2×2 factorial design.
BASE_CELL_SPECS = [
    ("affine_second_order", False),
    ("affine_second_order", True),
    ("conformal_first_integral", False),
    ("conformal_first_integral", True),
]

# The final fairness check: affine-specific nondimensionalization,
# repeated with RAR OFF and ON so that scaling and sampling remain separable.
FAIRNESS_CELL_SPECS = [
    ("affine_preconditioned", False),
    ("affine_preconditioned", True),
]

CELL_SPECS = BASE_CELL_SPECS + (FAIRNESS_CELL_SPECS if RUN_AFFINE_FAIRNESS else [])

results = []
cell_outputs = {}
for representation, rar_on in CELL_SPECS:
    print("\n" + "="*90)
    print(f"RUNNING: representation={representation} | RAR={'ON' if rar_on else 'OFF'}")
    print("="*90)
    out = train_one_cell(representation, rar_on, seed=SEED, verbose=True)
    metrics, arrays, history, stage_records, run_dir = out
    results.append(metrics)
    cell_outputs[(representation, rar_on)] = out
    display(pd.DataFrame([metrics]))
    gc.collect()

ablation_df = pd.DataFrame(results)
ablation_df.to_csv(RUN_DIR / "ablation_all_results.csv", index=False)
print("\nCompleted factorial + fairness-control experiment.")
display(ablation_df[[
    "representation", "rar", "eta_lambda_rel_L2", "x_lambda_rel_L2",
    "max_R_eta", "max_R_x", "max_R_lambda",
    "max_momentum_residual", "max_kappa_residual",
    "wall_seconds", "collocation_points_final"
]])



RUNNING: representation=affine_second_order | RAR=OFF
stage=0 epoch=1/1400 points=258 loss=9.109e+05
stage=0 epoch=700/1400 points=258 loss=6.768e+05
stage=0 epoch=1400/1400 points=258 loss=5.778e+05
stage=1 epoch=1/1400 points=322 loss=5.724e+05
stage=1 epoch=700/1400 points=322 loss=4.487e+05
stage=1 epoch=1400/1400 points=322 loss=3.733e+05
stage=2 epoch=1/1400 points=386 loss=3.701e+05
stage=2 epoch=700/1400 points=386 loss=3.000e+05
stage=2 epoch=1400/1400 points=386 loss=1.748e+08
stage=3 epoch=1/1400 points=450 loss=6.366e+08
stage=3 epoch=700/1400 points=450 loss=6.433e+06
stage=3 epoch=1400/1400 points=450 loss=4.228e+06
stage=4 epoch=1/1400 points=514 loss=4.225e+06
stage=4 epoch=700/1400 points=514 loss=2.696e+05
stage=4 epoch=1400/1400 points=514 loss=2.639e+05


,x_eta_rel_L2,lambda_eta_rel_L2,eta_lambda_rel_L2,x_lambda_rel_L2,max_R_eta,max_R_x,max_R_lambda,max_momentum_residual,max_kappa_residual,min_abs_dlambda_deta,...,adam_epochs_total,lbfgs_iters,reference_raw_vs_reduced_eta_max,reference_raw_vs_reduced_x_max,reference_raw_vs_reduced_lambda_max,eta_scale,x_scale,delta_lambda,normalized_eta_initial_slope,normalized_x_initial_slope
0,NaN,NaN,0.094504,0.21218,32.467385,31.217593,NaN,1.677444,4.093209,NaN,...,7000,220,0.0,9.582001e-10,1.983606e-09,NaN,NaN,NaN,NaN,NaN



RUNNING: representation=affine_second_order | RAR=ON
stage=0 epoch=1/1400 points=258 loss=9.109e+05
stage=0 epoch=700/1400 points=258 loss=6.768e+05
stage=0 epoch=1400/1400 points=258 loss=5.778e+05
stage=1 epoch=1/1400 points=322 loss=1.494e+06
stage=1 epoch=700/1400 points=322 loss=1.190e+06
stage=1 epoch=1400/1400 points=322 loss=9.636e+05
stage=2 epoch=1/1400 points=386 loss=9.080e+05
stage=2 epoch=700/1400 points=386 loss=8.330e+05
stage=2 epoch=1400/1400 points=386 loss=6.945e+05
stage=3 epoch=1/1400 points=450 loss=6.130e+05
stage=3 epoch=700/1400 points=450 loss=4.393e+05
stage=3 epoch=1400/1400 points=450 loss=3.472e+05
stage=4 epoch=1/1400 points=514 loss=3.039e+05
stage=4 epoch=700/1400 points=514 loss=2.341e+05
stage=4 epoch=1400/1400 points=514 loss=1.860e+05


,x_eta_rel_L2,lambda_eta_rel_L2,eta_lambda_rel_L2,x_lambda_rel_L2,max_R_eta,max_R_x,max_R_lambda,max_momentum_residual,max_kappa_residual,min_abs_dlambda_deta,...,adam_epochs_total,lbfgs_iters,reference_raw_vs_reduced_eta_max,reference_raw_vs_reduced_x_max,reference_raw_vs_reduced_lambda_max,eta_scale,x_scale,delta_lambda,normalized_eta_initial_slope,normalized_x_initial_slope
0,NaN,NaN,0.152512,0.441962,34.537053,33.110083,NaN,1.163942,3.877482,NaN,...,7000,220,0.0,9.582001e-10,1.983606e-09,NaN,NaN,NaN,NaN,NaN



RUNNING: representation=conformal_first_integral | RAR=OFF
stage=0 epoch=1/1400 points=258 loss=9.802e-01
stage=0 epoch=700/1400 points=258 loss=6.073e-06
stage=0 epoch=1400/1400 points=258 loss=2.405e-06
stage=1 epoch=1/1400 points=322 loss=2.444e-06
stage=1 epoch=700/1400 points=322 loss=3.441e-06
stage=1 epoch=1400/1400 points=322 loss=2.832e-06
stage=2 epoch=1/1400 points=386 loss=2.833e-06
stage=2 epoch=700/1400 points=386 loss=1.811e-06
stage=2 epoch=1400/1400 points=386 loss=1.965e-06
stage=3 epoch=1/1400 points=450 loss=1.973e-06
stage=3 epoch=700/1400 points=450 loss=1.506e-06
stage=3 epoch=1400/1400 points=450 loss=9.656e-07
stage=4 epoch=1/1400 points=514 loss=9.634e-07
stage=4 epoch=700/1400 points=514 loss=5.069e-07
stage=4 epoch=1400/1400 points=514 loss=7.120e-07


,x_eta_rel_L2,lambda_eta_rel_L2,eta_lambda_rel_L2,x_lambda_rel_L2,max_R_eta,max_R_x,max_R_lambda,max_momentum_residual,max_kappa_residual,min_abs_dlambda_deta,...,adam_epochs_total,lbfgs_iters,reference_raw_vs_reduced_eta_max,reference_raw_vs_reduced_x_max,reference_raw_vs_reduced_lambda_max,eta_scale,x_scale,delta_lambda,normalized_eta_initial_slope,normalized_x_initial_slope
0,0.000104,0.000212,0.000088,0.00017,NaN,0.000573,0.001195,0.007815,0.023161,0.060634,...,7000,220,0.0,9.582001e-10,1.983606e-09,NaN,NaN,NaN,NaN,NaN


... (4 outputs left out)


RUNNING: representation=affine_preconditioned | RAR=ON
stage=0 epoch=1/1400 points=258 loss=2.213e+03
stage=0 epoch=700/1400 points=258 loss=2.180e+02
stage=0 epoch=1400/1400 points=258 loss=7.617e+03
stage=1 epoch=1/1400 points=322 loss=1.070e+04
stage=1 epoch=700/1400 points=322 loss=8.373e+05
stage=1 epoch=1400/1400 points=322 loss=6.572e+05
stage=2 epoch=1/1400 points=386 loss=2.239e+08
stage=2 epoch=700/1400 points=386 loss=2.958e+05
stage=2 epoch=1400/1400 points=386 loss=2.859e+05
stage=3 epoch=1/1400 points=450 loss=6.773e+07
stage=3 epoch=700/1400 points=450 loss=7.992e+04
stage=3 epoch=1400/1400 points=450 loss=6.939e+04
stage=4 epoch=1/1400 points=514 loss=2.343e+08
stage=4 epoch=700/1400 points=514 loss=2.191e+03
stage=4 epoch=1400/1400 points=514 loss=2.182e+03


,x_eta_rel_L2,lambda_eta_rel_L2,eta_lambda_rel_L2,x_lambda_rel_L2,max_R_eta,max_R_x,max_R_lambda,max_momentum_residual,max_kappa_residual,min_abs_dlambda_deta,...,adam_epochs_total,lbfgs_iters,reference_raw_vs_reduced_eta_max,reference_raw_vs_reduced_x_max,reference_raw_vs_reduced_lambda_max,eta_scale,x_scale,delta_lambda,normalized_eta_initial_slope,normalized_x_initial_slope
0,NaN,NaN,0.072804,0.073565,0.904603,1.256639,NaN,0.970912,0.766235,NaN,...,7000,220,0.0,9.582001e-10,1.983606e-09,13.193938,12.8,0.8,1.0,1.0



Completed factorial + fairness-control experiment.


,representation,rar,eta_lambda_rel_L2,x_lambda_rel_L2,max_R_eta,max_R_x,max_R_lambda,max_momentum_residual,max_kappa_residual,wall_seconds,collocation_points_final
0,affine_second_order,False,0.094504,0.212180,32.467385,31.217593,NaN,1.677444e+00,4.093209e+00,275.192840,514
1,affine_second_order,True,0.152512,0.441962,34.537053,33.110083,NaN,1.163942e+00,3.877482e+00,282.468489,514
2,conformal_first_integral,False,0.000088,0.000170,NaN,0.000573,0.001195,7.815028e-03,2.316106e-02,101.501571,514
3,conformal_first_integral,True,0.000126,0.000216,NaN,0.000471,0.001400,7.927516e-03,1.737090e-02,104.703757,514
4,affine_preconditioned,False,9.671038,4.420038,5094.413054,445.800621,NaN,7.894186e+06,1.290576e+09,280.518462,514
5,affine_preconditioned,True,0.072804,0.073565,0.904603,1.256639,NaN,9.709121e-01,7.662353e-01,285.253425,514


## 9. Factorial summary and diagnostic plots

The primary paper table is the four-cell comparison in the common affine representation.

We also calculate descriptive effect ratios:

- **RAR effect within affine:** affine RAR ON vs OFF;
- **RAR effect within conformal:** conformal RAR ON vs OFF;
- **representation effect without RAR:** conformal OFF vs affine OFF;
- **representation effect with RAR:** conformal ON vs affine ON.

These are descriptive ratios from this controlled run. They are not treated as universal scaling laws.

In [ ]:
# ============================================================
# 10. FACTORIAL + FAIRNESS SUMMARY
# ============================================================

pivot_eta = ablation_df.pivot(index="representation", columns="rar", values="eta_lambda_rel_L2")
pivot_x = ablation_df.pivot(index="representation", columns="rar", values="x_lambda_rel_L2")

print("Common affine-parameter errors: eta(lambda)")
display(pivot_eta)
print("Common affine-parameter errors: x(lambda)")
display(pivot_x)


def ratio(a, b):
    return float(a / b) if np.isfinite(a) and np.isfinite(b) and b != 0 else np.nan

lookup = {
    (r, bool(rr)): ablation_df[(ablation_df["representation"] == r) & (ablation_df["rar"] == bool(rr))].iloc[0]
    for r, rr in CELL_SPECS
}

effect_rows = []

def add_effect(label, numerator_key, denominator_key):
    if numerator_key in lookup and denominator_key in lookup:
        a = lookup[numerator_key]
        b = lookup[denominator_key]
        effect_rows.append({
            "comparison": label,
            "eta_lambda_ratio": ratio(a["eta_lambda_rel_L2"], b["eta_lambda_rel_L2"]),
            "x_lambda_ratio": ratio(a["x_lambda_rel_L2"], b["x_lambda_rel_L2"]),
        })

add_effect("RAR effect | raw affine (ON/OFF)",
           ("affine_second_order", True), ("affine_second_order", False))
add_effect("RAR effect | conformal (ON/OFF)",
           ("conformal_first_integral", True), ("conformal_first_integral", False))
add_effect("Representation effect | RAR OFF (conformal/raw affine)",
           ("conformal_first_integral", False), ("affine_second_order", False))
add_effect("Representation effect | RAR ON (conformal/raw affine)",
           ("conformal_first_integral", True), ("affine_second_order", True))
if RUN_AFFINE_FAIRNESS:
    add_effect("Affine preconditioning effect | RAR OFF (precond/raw)",
               ("affine_preconditioned", False), ("affine_second_order", False))
    add_effect("Affine preconditioning effect | RAR ON (precond/raw)",
               ("affine_preconditioned", True), ("affine_second_order", True))
    add_effect("Representation effect | RAR OFF (conformal/preconditioned)",
               ("conformal_first_integral", False), ("affine_preconditioned", False))
    add_effect("Representation effect | RAR ON (conformal/preconditioned)",
               ("conformal_first_integral", True), ("affine_preconditioned", True))

effects = pd.DataFrame(effect_rows)
effects.to_csv(RUN_DIR / "factor_effects.csv", index=False)
display(effects)

# Publication-friendly bar chart on common affine errors.
plot_df = ablation_df.copy()
plot_df["label"] = plot_df.apply(lambda r: f"{r['representation']}\nRAR={'on' if r['rar'] else 'off'}", axis=1)
fig = plt.figure(figsize=(11, 6))
x = np.arange(len(plot_df))
width = 0.36
plt.bar(x - width/2, plot_df["eta_lambda_rel_L2"], width, label=r"$\eta(\lambda)$ relative $L^2$")
plt.bar(x + width/2, plot_df["x_lambda_rel_L2"], width, label=r"$x(\lambda)$ relative $L^2$")
plt.yscale("log")
plt.xticks(x, plot_df["label"], rotation=25, ha="right")
plt.ylabel("Relative $L^2$ error (log scale)")
plt.title("Controlled ablation and affine-preconditioning fairness check")
plt.grid(axis="y", alpha=0.3)
plt.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "ablation_with_fairness_control.png", dpi=240, bbox_inches="tight")
plt.close(fig)


Common affine-parameter errors: eta(lambda)


rar,False,True
representation,,
affine_preconditioned,9.671038,0.072804
affine_second_order,0.094504,0.152512
conformal_first_integral,0.000088,0.000126


Common affine-parameter errors: x(lambda)


rar,False,True
representation,,
affine_preconditioned,4.420038,0.073565
affine_second_order,0.212180,0.441962
conformal_first_integral,0.000170,0.000216


,comparison,eta_lambda_ratio,x_lambda_ratio
0,RAR effect | raw affine (ON/OFF),1.613811,2.082960
1,RAR effect | conformal (ON/OFF),1.426982,1.267647
2,Representation effect | RAR OFF (conformal/raw...,0.000932,0.000803
3,Representation effect | RAR ON (conformal/raw ...,0.000824,0.000489
4,Affine preconditioning effect | RAR OFF (preco...,102.334551,20.831551
5,Affine preconditioning effect | RAR ON (precon...,0.477363,0.166450
6,Representation effect | RAR OFF (conformal/pre...,0.000009,0.000039
7,Representation effect | RAR ON (conformal/prec...,0.001727,0.002937


## 10. Sanity checks specific to the factorial design

Before interpreting the ablation, this notebook checks:

1. all four cells used the same seed;
2. all four cells used the same network architecture;
3. all four cells used the same total Adam epoch budget;
4. all four cells reached the same final collocation count;
5. the raw DOP853 and reduced first-integral references agree to numerical precision;
6. the common affine validation grid is identical.

If any of these checks fail, **do not interpret the 2×2 result**. Fix the bookkeeping first.

In [ ]:
# ============================================================
# 11. DESIGN SANITY CHECKS
# ============================================================

assert ablation_df["seed"].nunique() == 1
assert ablation_df["seed"].iloc[0] == SEED
assert set(ablation_df["adam_epochs_total"]) == {CFG["ADAM_EPOCHS_PER_STAGE"] * (CFG["N_STAGES"] + 1)}
assert set(ablation_df["collocation_points_final"]) == {CFG["N_INITIAL"] + 2 + CFG["N_ADD_PER_STAGE"] * CFG["N_STAGES"]}
expected_cells = 4 + (2 if RUN_AFFINE_FAIRNESS else 0)
assert len(ablation_df) == expected_cells
assert REF["raw_reduced_x_max_abs"] < 1e-7, "Independent reference consistency failed."
assert REF["raw_reduced_lambda_max_abs"] < 1e-7, "Independent reference consistency failed."

# Shared initial-state bookkeeping: all cells use the same parameter state dictionary.
if RUN_AFFINE_FAIRNESS:
    for rar_flag in (False, True):
        row = ablation_df[(ablation_df["representation"] == "affine_preconditioned") & (ablation_df["rar"] == rar_flag)].iloc[0]
        assert np.isfinite(row["eta_scale"]) and np.isfinite(row["x_scale"])
        assert abs(row["normalized_eta_initial_slope"]) - 1.0 < 1e-12
        assert abs(row["normalized_x_initial_slope"]) - 1.0 < 1e-12

print("PASS: factorial cell count =", len(ablation_df))
print("PASS: same seed:", SEED)
print("PASS: same final collocation count:", int(ablation_df["collocation_points_final"].iloc[0]))
print("PASS: same total Adam epochs:", int(ablation_df["adam_epochs_total"].iloc[0]))
print("PASS: raw/reduced reference consistency is within tolerance.")
if RUN_AFFINE_FAIRNESS:
    print("PASS: affine preconditioning is based only on prescribed initial data and produces O(1) normalized slopes.")


PASS: factorial cell count = 6
PASS: same seed: 42
PASS: same final collocation count: 514
PASS: same total Adam epochs: 7000
PASS: raw/reduced reference consistency is within tolerance.
PASS: affine preconditioning is based only on prescribed initial data and produces O(1) normalized slopes.


## 11. Optional second-seed confirmation

The primary factorial result is intentionally fixed to one seed so that representation and RAR are the only experimental factors.

If the 2×2 differences are close enough that you want an initialization check, set `RUN_SECOND_SEED_CONFIRMATION = True` at the top and rerun from a fresh Colab runtime. The confirmation repeats the same four cells at seed 100.

This confirmation is **not** a replacement for the earlier five-seed robustness result; it is only a check that the factorial pattern is not peculiar to seed 42.

In [ ]:
# ============================================================
# 12. OPTIONAL SECOND-SEED CONFIRMATION
# ============================================================

if RUN_SECOND_SEED_CONFIRMATION:
    confirmation_rows = []
    confirmation_seed = 100
    shared_state_2 = make_shared_initial_state(confirmation_seed)
    old_state = SHARED_INIT_STATE
    SHARED_INIT_STATE = shared_state_2
    try:
        for representation, rar_on in CELL_SPECS:
            out = train_one_cell(representation, rar_on, seed=confirmation_seed, verbose=False)
            metrics = out[0]
            confirmation_rows.append(metrics)
        confirmation_df = pd.DataFrame(confirmation_rows)
        confirmation_df.to_csv(RUN_DIR / "second_seed_confirmation.csv", index=False)
        display(confirmation_df[[
            "representation", "rar", "eta_lambda_rel_L2", "x_lambda_rel_L2",
            "max_R_eta", "max_R_x", "max_R_lambda"
        ]])
    finally:
        SHARED_INIT_STATE = old_state
else:
    print("Second-seed confirmation disabled; primary factorial result uses seed", SEED)

Second-seed confirmation disabled; primary factorial result uses seed 42


## 12. Evidence report for Paper 1

This cell creates a machine-readable and human-readable evidence summary. It deliberately avoids calling any cell a "winner". The paper should report the measured factor effects and their experimental conditions, then interpret them cautiously.

In [ ]:
# ============================================================
# 13. MANUSCRIPT-READY EVIDENCE PACK
# ============================================================

paper_table = ablation_df[[
    "representation", "rar", "eta_lambda_rel_L2", "x_lambda_rel_L2",
    "x_eta_rel_L2", "lambda_eta_rel_L2", "max_R_eta", "max_R_x",
    "max_R_lambda", "max_momentum_residual", "max_kappa_residual",
    "wall_seconds", "collocation_points_final", "adam_epochs_total", "lbfgs_iters",
    "eta_scale", "x_scale", "delta_lambda"
]].copy()

paper_table.to_csv(RUN_DIR / "Paper1_Table_Ablation_With_Fairness_Control.csv", index=False)

report_lines = []
report_lines.append("# Paper 1 — Controlled 2×2 Ablation + Affine Fairness Control")
report_lines.append("")
report_lines.append(f"Run ID: {RUN_ID}")
report_lines.append(f"Seed: {SEED}")
report_lines.append(f"Profile: {PROFILE}")
report_lines.append("")
report_lines.append("## Controlled factors")
report_lines.append("")
report_lines.append("Primary factorial: representation (affine second-order vs conformal first-integral) × sampling (fixed vs RAR).")
report_lines.append("Fairness control: affine second-order equations written in dimensionless affine/output variables using scales derived only from prescribed initial data.")
report_lines.append("")
report_lines.append("## Measured errors")
report_lines.append("")
report_lines.append("| Representation | RAR | eta(lambda) rel L2 | x(lambda) rel L2 | max p residual | max kappa residual |")
report_lines.append("|---|---:|---:|---:|---:|---:|")
for _, row in paper_table.iterrows():
    report_lines.append(
        f"| {row['representation']} | {bool(row['rar'])} | "
        f"{row['eta_lambda_rel_L2']:.6e} | {row['x_lambda_rel_L2']:.6e} | "
        f"{row['max_momentum_residual']:.6e} | {row['max_kappa_residual']:.6e} |"
    )
report_lines.append("")
report_lines.append("## Reference consistency")
report_lines.append("")
report_lines.append(f"Raw-vs-reduced reference max |eta| discrepancy: {REF['raw_reduced_eta_max_abs']:.6e}")
report_lines.append(f"Raw-vs-reduced reference max |x| discrepancy: {REF['raw_reduced_x_max_abs']:.6e}")
report_lines.append(f"Raw-vs-reduced reference max |lambda| discrepancy: {REF['raw_reduced_lambda_max_abs']:.6e}")
report_lines.append("")
report_lines.append("## Interpretation discipline")
report_lines.append("")
report_lines.append("The fairness control tests whether straightforward affine-specific nondimensionalization materially closes the performance gap. A remaining large conformal advantage strengthens the representation-level interpretation, but it is not a proof of universal superiority outside the tested benchmark and architecture family.")

with open(RUN_DIR / "Paper1_Fairness_Control_Evidence_Report.md", "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print("Saved evidence files in", RUN_DIR)


Saved evidence files in paper1_results/20260925T230536Z_c8ce5c_2x2_ablation


In [ ]:
# ============================================================
# 13. APPEND ALL FACTORIAL + FAIRNESS CELLS TO GLOBAL REGISTRY
# ============================================================

for _, row in ablation_df.iterrows():
    append_registry_row(row.to_dict())

print("Global registry updated:", RESULTS_ROOT / "run_registry.csv")
print(pd.read_csv(RESULTS_ROOT / "run_registry.csv").tail(len(ablation_df)))


Global registry updated: paper1_results/run_registry.csv
   status  profile  seed            representation    rar  x_eta_rel_L2  \
0     NaN      NaN    42       affine_second_order  False           NaN   
1     NaN      NaN    42       affine_second_order   True           NaN   
2     NaN      NaN    42  conformal_first_integral  False      0.000104   
3     NaN      NaN    42  conformal_first_integral   True      0.000076   
4     NaN      NaN    42     affine_preconditioned  False           NaN   
5     NaN      NaN    42     affine_preconditioned   True           NaN   

   lambda_eta_rel_L2  eta_lambda_rel_L2  x_lambda_rel_L2    max_R_eta  ...  \
0                NaN           0.094504         0.212180    32.467385  ...   
1                NaN           0.152512         0.441962    34.537053  ...   
2           0.000212           0.000088         0.000170          NaN  ...   
3           0.000306           0.000126         0.000216          NaN  ...   
4                NaN       

## 13. Export the complete run for Colab

Colab can download a file directly but not a directory reliably. This cell packages the **entire `paper1_results/` tree** into one ZIP. Previous runs are preserved.

Run this cell when the experiment is complete and download the resulting ZIP to your computer.

In [ ]:
# ============================================================
# 14. PACKAGE THE COMPLETE PAPER 1 RESULTS TREE
# ============================================================

colab_root = Path("/content") if Path("/content").exists() else Path.cwd()
archive_base = colab_root / f"Paper1_2x2_Ablation_{RUN_ID}"
zip_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=str(colab_root),
    base_dir="paper1_results",
)

# Manifest for easy inspection outside Colab.
manifest_path = RUN_DIR / "EXPORT_MANIFEST.txt"
with open(manifest_path, "w", encoding="utf-8") as f:
    for p in sorted(RESULTS_ROOT.rglob("*")):
        if p.is_file():
            f.write(str(p.relative_to(RESULTS_ROOT)) + "\n")

print("Created ZIP:", zip_path)
print("ZIP size MB:", round(Path(zip_path).stat().st_size / 1024**2, 2))
print("Manifest:", manifest_path)

try:
    from google.colab import files
    files.download(zip_path)
    print("Colab download requested. If the browser does not start it, use the left Files panel and download the ZIP manually.")
except Exception:
    print("Not running inside Colab. The ZIP is still available at:", zip_path)

Created ZIP: /content/Paper1_2x2_Ablation_20260925T230536Z_c8ce5c.zip
ZIP size MB: 5.1
Manifest: paper1_results/20260925T230536Z_c8ce5c_2x2_ablation/EXPORT_MANIFEST.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Colab download requested. If the browser does not start it, use the left Files panel and download the ZIP manually.


# 14. What to do after this run

### The new fairness question

The four original cells answer whether the **conformal/first-integral representation** behaves differently from the affine/second-order representation and whether RAR explains that difference.

The two additional cells answer a stricter question:

> Does the affine baseline improve substantially when its variables and second-order residuals are explicitly nondimensionalized using scales known before training?

If the preconditioned affine errors approach the conformal errors, the paper must weaken its representation-level claim. If a large gap remains, the evidence for the conformal representation becomes considerably stronger.

### Interpretation discipline

Do not select a "winner" from a single metric. Compare:

- common affine trajectory errors \(\eta(\lambda),x(\lambda)\);
- native residuals;
- invariant residuals;
- training stability and wall time;
- and whether the same qualitative pattern appears under both RAR conditions.

Treat the result as evidence for this benchmark and architecture family, not as a universal theorem about PINNs or reparameterizations.

### Export

Use the final packaging cell to download the complete `paper1_results/` tree as a ZIP. Preserve the previous runs; do not delete the results directory.
